# Deleting an index

If you have an index that has gone stale, perhaps it's time to delete it!
Deleting an index will also delete all of the data it contains,
so be cautious when doing this in your own projects!

In [1]:
import os

from dotenv import load_dotenv
from pinecone import Pinecone

load_dotenv()

# Initialize the Pinecone client with your API key
pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))

need_to_delete_indexes = [
    "my-first-index",
    "pinecone-datacamp",
    "datacamp-index",
    "dotproduct-index",
]

pinecone_indexes = pc.list_indexes()

for to_delete_index in need_to_delete_indexes:
    if to_delete_index in [index.name for index in pinecone_indexes]:
        pc.delete_index(to_delete_index)
        print(f"{to_delete_index} successfully deleted.")
    else:
        print(f"{to_delete_index} not in index list.")

my-first-index successfully deleted.
pinecone-datacamp not in index list.
datacamp-index successfully deleted.
dotproduct-index not in index list.


# Your first Pinecone index

With your Pinecone client initialized, you're all set to begin creating an index!
Indexes are used to store records, including the vectors and associated metadata,
as well as serving queries and other manipulations.
As you progress through the course,
you'll see how these different steps build up
to a modern AI system built on a vector database.

In [2]:
import os

from dotenv import load_dotenv

# Import ServerlessSpec
from pinecone import Pinecone, ServerlessSpec

load_dotenv()

# Initialize the Pinecone client with your API key
pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))

# Create your Pinecone index
pc.create_index(
    name="my-first-index",
    dimension=256,
    spec=ServerlessSpec(
        cloud="aws",
        region="us-east-1",
    ),
)

pc.list_indexes()

IndexList([<name='my-first-index', ready=True>])

# Connecting to an index

To begin ingesting vectors and performing vector manipulations
in your newly-created Pinecone index,
you'll first need to connect to it!
The resulting index object has a number of methods for ingesting, manipulating,
and exploring the contents of the index in Python.

In [3]:
from pinecone_helper import get_pinecone_client

pc = get_pinecone_client()

# Connect to the index
index = pc.Index("my-first-index")

# Print the index statistics
print(index.describe_index_stats())

DescribeIndexStatsResponse(dimension=256, total_vector_count=0, metric='cosine', namespaces=0)


# Checking dimensionality

You now have the know-how to begin ingesting vectors into a new Pinecone index!
Before you jump in, you should check that your vectors are compatible
with the dimensionality of your new index.

A list of dictionaries containing records to ingest has been provided as vectors.
Here's a preview of its structure:
```python
vectors = [
    {
        "id": "0",
        "values": [0.025525547564029694, ..., 0.0188823901116848]
        "metadata": {"genre": "action", "year": 2024}
    },
        ...,
]
```

In [4]:
from pinecone import ServerlessSpec
from pinecone_helper import get_pinecone_client

pc = get_pinecone_client()
index_name = "datacamp-index"

if index_name not in [index.name for index in pc.list_indexes()]:
    # Create your Pinccone index
    pc.create_index(
        name="datacamp-index",
        dimension=1536,
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1",
        ),
    )

In [5]:
from DATA import vectors

# Check that each vector has a dimensionality of 1536
vector_items = [len(vector["values"]) == 1536 for vector in vectors]
print(all(vector_items))

True


# Ingesting vectors with metadata

It's ingesting time! You'll be ingesting vectors,
which is a list of dictionaries containing the vector values, IDs,
and associated metadata.
They're already been provided in a format
that can be directly inserted into the index without further manipulation.

Here's another reminder about the structure of vectors.

```python
vectors = [
    {
        "id": "0",
        "values": [0.025525547564029694, ..., 0.0188823901116848]
        "metadata": {"genre": "action", "year": 2024}
    },
        ...,
]
```

In [6]:
from DATA import vectors
from pinecone_helper import get_pinecone_client

pc = get_pinecone_client()

# Connect to your index
index = pc.Index("datacamp-index")

# Ingets the vectors and metadata
index.upsert(vectors=vectors)

# Print the index statistics
print(index.describe_index_stats())

DescribeIndexStatsResponse(dimension=1536, total_vector_count=100, metric='cosine', namespaces=1)


# Fetching vectors

In this exercise, you've been provided with a list of ids containing IDs
of different records in your 'datacamp-index' index.
You'll use these IDs to retrieve the associated records and explore their metadata.

In [7]:
from pinecone_helper import get_pinecone_client

pc = get_pinecone_client()

index = pc.Index("datacamp-index")
ids = ["2", "5", "8"]

# Fetch the vectors from the connected Pinecone index
fetched_vectors = index.fetch(ids=ids)

# Extract the metadata from each result in feteched_vectors
metadatas = [fetched_vectors["vectors"][id]["metadata"] for id in ids]
print(metadatas)


[{'genre': 'comedy', 'year': 2004}, {'genre': 'thriller', 'year': 2001}, {'genre': 'action', 'year': 2017}]


# Returning the most similar vectors

Querying vectors is foundational to so many AI applications.
It involves embedding a user input, comparing it to the vectors in the database,
and returning the most similar vectors.

In this exercise, you've been provided with a mystery vector called vector
and you'll use it to query your index called 'datacamp-index'.

In [8]:
from DATA import vector
from pinecone_helper import get_index

index = get_index()

# Retrieve the top three most similar records
query_result = index.query(
    vector=vector,
    top_k=3,
)

print(query_result)

QueryResponse(matches=[ScoredVector(id='0', score=0.999999881, values=[]), ScoredVector(id='89', score=0.0631826222, values=[]), ScoredVector(id='42', score=0.061817944, values=[])], namespace='', usage=Usage(read_units=1, write_units=None), response_info=ResponseInfo(raw_headers={'date': 'Sun, 27 Sep 2026 03:48:43 GMT', 'content-type': 'application/json', 'content-length': '214', 'connection': 'keep-alive', 'x-pinecone-max-indexed-lsn': '1', 'x-pinecone-request-latency-ms': '257', 'x-envoy-upstream-service-time': '39', 'x-pinecone-response-duration-ms': '258', 'grpc-status': '0', 'server': 'envoy'}))


# Changing distance metrics

By default, Pinecone indexes using the cosine similarity distance metric
to compute similarity scores between vectors,
which are used when querying to find the most similar vectors.
Pinecone also supports other distance metrics,
including Euclidean distance and the dot product.

The distance metric is set when the index is created,
and can't be changed afterwards. In this exercise,
you'll practice creating an index that uses the dot product distance metric.

In [1]:
from pinecone import ServerlessSpec
from pinecone_helper import get_pinecone_client

pc = get_pinecone_client()

# Create an index that uses the dot product distance metric
pc.create_index(
    name="dotproduct-index",
    dimension=1536,
    metric="dotproduct",
    spec=ServerlessSpec(
        cloud="aws",
        region="us-east-1",
    ),
)

# Print a list of your indexes
print(pc.list_indexes())

IndexList([<name='my-first-index', ready=True>, <name='datacamp-index', ready=True>, <name='dotproduct-index', ready=True>])
